# URL Phishing Detection — LegitPhish Investigation

## Objective

In this notebook, I investigate an additional URL dataset following the limitations identified during my PhiUSIIL experiments.

My earlier URL classifier achieved strong validation performance, but further analysis revealed that structural differences between legitimate and phishing URLs substantially influenced its predictions.

I now examine the LegitPhish dataset to determine whether it provides more realistic URL diversity and a more credible basis for evaluating phishing detection.

I begin by assessing dataset quality, label definitions, URL structure, and potential overlap with PhiUSIIL.

I then evaluate the existing classifier on compatible, independently sourced examples before deciding whether additional model development is justified.

My goal is to build a CPU-friendly URL phishing detector whose performance is supported by meaningful evaluation rather than dataset-specific shortcuts.

## 1. Initial Dataset Exploration

I begin by examining the structure and quality of the LegitPhish dataset.

My previous investigation revealed that the PhiUSIIL dataset contained strong structural differences between legitimate and phishing URLs, which inflated the apparent performance of my classifier.

Before developing another model, I want to understand the characteristics of this new dataset.

I examine its dimensions, feature columns, classification labels, missing values, and duplicate URLs.

I also distinguish original URL strings from precomputed features because my eventual detector must extract its features directly from URLs without visiting websites.

In [3]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

RANDOM_STATE = 42

# I configure portable project paths.
PROJECT_ROOT = Path.cwd().resolve().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROCESSED_DATA_DIR / "splits"

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

LEGITPHISH_PATH = (
    RAW_DATA_DIR / "url_features_extracted1.csv"
)

# I verify that the expected dataset exists.
assert LEGITPHISH_PATH.is_file(), (
    f"Dataset not found: {LEGITPHISH_PATH}"
)

print("LEGITPHISH ENVIRONMENT")
print("Project root:", PROJECT_ROOT)
print("Dataset:", LEGITPHISH_PATH.name)

print(
    "Dataset size:",
    f"{LEGITPHISH_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)

print("Environment verification: PASSED")

LEGITPHISH ENVIRONMENT
Project root: C:\Users\JONATHAN\Desktop\WORK\spam-phishing-detector
Dataset: url_features_extracted1.csv
Dataset size: 8.55 MB
Environment verification: PASSED


In [5]:
# I load the LegitPhish dataset without modifying its contents.
start_time = time.perf_counter()

legitphish_df = pd.read_csv(
    LEGITPHISH_PATH,
    low_memory=False
)

load_time = time.perf_counter() - start_time

print("LEGITPHISH DATASET OVERVIEW")

print("Shape:", legitphish_df.shape)
print(f"Loading time: {load_time:.2f} seconds")

print("\nCOLUMN NAMES")
print(legitphish_df.columns.tolist())

print("\nDATA TYPES")
print(legitphish_df.dtypes.to_string())

print("\nFIRST FIVE RECORDS")
display(legitphish_df.head())

LEGITPHISH DATASET OVERVIEW
Shape: (101219, 18)
Loading time: 0.19 seconds

COLUMN NAMES
['URL', 'url_length', 'has_ip_address', 'dot_count', 'https_flag', 'url_entropy', 'token_count', 'subdomain_count', 'query_param_count', 'tld_length', 'path_length', 'has_hyphen_in_domain', 'number_of_digits', 'tld_popularity', 'suspicious_file_extension', 'domain_name_length', 'percentage_numeric_chars', 'ClassLabel']

DATA TYPES
URL                              str
url_length                     int64
has_ip_address                 int64
dot_count                      int64
https_flag                     int64
url_entropy                  float64
token_count                    int64
subdomain_count                int64
query_param_count              int64
tld_length                     int64
path_length                    int64
has_hyphen_in_domain           int64
number_of_digits               int64
tld_popularity                 int64
suspicious_file_extension      int64
domain_name_length     

,URL,url_length,has_ip_address,dot_count,https_flag,url_entropy,token_count,subdomain_count,query_param_count,tld_length,path_length,has_hyphen_in_domain,number_of_digits,tld_popularity,suspicious_file_extension,domain_name_length,percentage_numeric_chars,ClassLabel
0,https://keraekken-loagginnusa.godaddysites.com/,47,0,2,1,4.250669,6,1,1,3,1,1,0,1,0,12,0.0,0.0
1,https://metamsk01lgiix.godaddysites.com/,40,0,2,1,4.196439,6,1,1,3,1,0,2,1,0,12,5.0,0.0
2,http://myglobaltech.in/,23,0,1,0,3.936180,5,0,1,2,1,0,0,0,0,12,0.0,0.0
3,http://djtool-for-spotify.com/,30,0,1,0,3.894740,5,0,1,3,1,1,0,1,0,18,0.0,0.0
4,https://scearmcoommunnlty.com/invent/freind/get,47,0,1,1,4.143127,7,0,1,3,18,0,0,1,0,17,0.0,0.0


In [7]:
# I examine missing values across all columns.
missing_summary = (
    legitphish_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("MISSING VALUES")

if (missing_summary > 0).any():
    print(missing_summary[missing_summary > 0])
else:
    print("No missing values detected.")


# I inspect the original classification labels.
print("\nORIGINAL LABEL DISTRIBUTION")

print(
    legitphish_df["ClassLabel"]
    .value_counts(dropna=False)
)


# I check for complete duplicate records.
print("\nEXACT DUPLICATE ROWS")

print(
    legitphish_df.duplicated().sum()
)


# I check for duplicate URL strings.
print("\nDUPLICATE URL STRINGS")

print(
    legitphish_df["URL"].duplicated().sum()
)


print("\nMISSING URL VALUES")

print(
    legitphish_df["URL"].isna().sum()
)

MISSING VALUES
ClassLabel    1
dtype: int64

ORIGINAL LABEL DISTRIBUTION
ClassLabel
0.0    63678
1.0    37540
NaN        1
Name: count, dtype: int64

EXACT DUPLICATE ROWS
346

DUPLICATE URL STRINGS
346

MISSING URL VALUES
0


## 2. Label Consistency and URL Structure Analysis

I investigate the data-quality issues identified during the initial exploration of LegitPhish.

The dataset contains one missing classification label and several duplicated URL records. I examine these records before deciding how to handle them during preprocessing.

I also investigate the structural characteristics of legitimate and phishing URLs.

My earlier PhiUSIIL experiments revealed that all legitimate URLs used HTTPS and had empty paths, creating strong dataset-specific shortcuts.

I now examine whether similar patterns exist in LegitPhish, particularly in HTTPS usage, URL paths, and query parameters.

This analysis will help me determine whether the dataset provides a more credible foundation for URL phishing detection.

In [10]:
# I identify URLs appearing more than once.
duplicate_mask = legitphish_df["URL"].duplicated(
    keep=False
)

duplicate_records = legitphish_df.loc[
    duplicate_mask,
    ["URL", "ClassLabel"]
].copy()

print("DUPLICATE URL ANALYSIS")

print(
    "Records in duplicate groups:",
    len(duplicate_records)
)

print(
    "Unique duplicated URLs:",
    duplicate_records["URL"].nunique()
)

print(
    "Additional duplicate records:",
    legitphish_df["URL"].duplicated().sum()
)

# I investigate whether identical URLs have conflicting labels.
url_label_counts = (
    legitphish_df
    .groupby("URL")["ClassLabel"]
    .nunique(dropna=True)
)

conflicting_urls = url_label_counts[
    url_label_counts > 1
]

print("\nLABEL CONSISTENCY")

print(
    "URLs with conflicting labels:",
    len(conflicting_urls)
)

print(
    "Records associated with conflicting URLs:",
    int(
        legitphish_df["URL"]
        .isin(conflicting_urls.index)
        .sum()
    )
)

DUPLICATE URL ANALYSIS
Records in duplicate groups: 648
Unique duplicated URLs: 302
Additional duplicate records: 346

LABEL CONSISTENCY
URLs with conflicting labels: 0
Records associated with conflicting URLs: 0


In [12]:
# I inspect the record with a missing classification label.
missing_label_records = legitphish_df[
    legitphish_df["ClassLabel"].isna()
]

print("MISSING LABEL ANALYSIS")

print(
    "Records with missing labels:",
    len(missing_label_records)
)

display(
    missing_label_records[
        ["URL", "ClassLabel"]
    ]
)

MISSING LABEL ANALYSIS
Records with missing labels: 1


,URL,ClassLabel
65118,https://www.chocolatleroux.com,NaN


In [14]:
from urllib.parse import urlsplit

# I extract structural characteristics directly from URL strings.
def inspect_url_structure(url):

    url = str(url).strip()

    parseable_url = (
        url if "://" in url
        else "http://" + url
    )

    try:
        parsed = urlsplit(parseable_url)

        return {
            "is_https": int(
                parsed.scheme.lower() == "https"
            ),
            "path_length": len(parsed.path),
            "has_path": int(bool(parsed.path)),
            "has_query": int(bool(parsed.query)),
            "url_length": len(url)
        }

    except ValueError:
        return {
            "is_https": 0,
            "path_length": 0,
            "has_path": 0,
            "has_query": 0,
            "url_length": len(url)
        }


# I retain only records with valid binary labels for this analysis.
structure_source = legitphish_df[
    legitphish_df["ClassLabel"].isin([0, 1])
].copy()

# I calculate URL characteristics without accessing websites.
url_structure = pd.DataFrame(
    [
        inspect_url_structure(url)
        for url in structure_source["URL"]
    ],
    index=structure_source.index
)

# I retain the original classification labels.
url_structure["label"] = (
    structure_source["ClassLabel"].astype(int)
)

# I summarize URL characteristics by class.
structure_summary = (
    url_structure
    .groupby("label")
    .agg(
        records=("label", "size"),
        https_rate=("is_https", "mean"),
        path_rate=("has_path", "mean"),
        query_rate=("has_query", "mean"),
        average_url_length=("url_length", "mean"),
        average_path_length=("path_length", "mean")
    )
)

structure_summary = structure_summary.rename(
    index={
        0: "Phishing",
        1: "Legitimate"
    }
)

print("LEGITPHISH URL STRUCTURE BY CLASS")

display(structure_summary.round(4))

LEGITPHISH URL STRUCTURE BY CLASS


,records,https_rate,path_rate,query_rate,average_url_length,average_path_length
label,,,,,,
Phishing,63678,0.0553,0.9998,0.0118,39.4119,12.6922
Legitimate,37540,0.9999,0.0160,0.0014,27.6809,0.4868


## 3. Dataset Suitability Assessment

I investigated LegitPhish as an alternative to PhiUSIIL after identifying substantial structural bias in my earlier URL classification experiments.

The dataset contains 101,219 records, including one missing classification label and 346 additional duplicate URL records. I found no conflicting labels among repeated URLs.

Although LegitPhish provides original URL strings and independently sourced examples, its structural characteristics revealed a similar limitation to PhiUSIIL.

Approximately 99.99% of legitimate URLs use HTTPS, while only 5.53% of phishing URLs do.

Additionally, only 1.60% of legitimate URLs have non-empty paths, compared with 99.98% of phishing URLs.

These differences suggest that a classifier could achieve strong performance by learning dataset collection patterns rather than generalizable phishing characteristics.

I therefore decided not to use LegitPhish as a straightforward replacement for PhiUSIIL or immediately combine the datasets for training.

This investigation reinforced the importance of evaluating the composition of both classes before accepting a phishing URL dataset for model development.